# 21 · 正式评测：系统定稿后，在留出集上只考一次

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → [21 正式评测 ✅] → 22 做成服务
```

> **这一章的代码在合并请求 #22、#23 里。** 留出集一共考了 8 次，花了约 ¥39，全部记在 MLflow 里。下面每一格都只读记录、在本机计算，**运行这本笔记不花钱**（第 8 步要读 8 次考试的全部记录，大约几分钟）。
>
> **这一章分两段写。** 第 1–6 步是考之前写的：公司里一次正式实验怎么做、怎么记录、架构是什么样、方案和版本怎么定。第 7 步起是考完以后填的结果。这样安排是有意的：**怎么考、怎么算赢，必须在看到结果之前定下来。**

## 背景：为什么 dev-200 的分数不能直接对外说

到第 20 章，所有分数都来自 dev-200 这 200 道题。可是我们的每一个决定，都是看着这 200 道题的分数做的：

- 第 16 章：看错题，决定做哪些工具
- 第 19 章：小考发现 agent 忽略基因证据，改了一句提示
- 第 20 章：消融实验，决定留"找相似病人"、删"查疾病介绍"；定稿前又加了"和基因排名合并"

每做一个这样的决定，系统就向这 200 道题多"贴合"了一点。这叫**对开发集过拟合**：分数里有一部分不是系统的真本事，而是"恰好适合这 200 道题"。

所以项目第一天（第 06 章）就把病例按论文分成了两份：一份开发用，一份**留出集**放着不动，4,360 个病人、840 篇论文，从来没有任何系统在上面跑过。系统定稿以后，在它上面只考一次，那个分数才是能对外说的。

## 这一章做的 9 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 公司里一次正式实验怎么做 | 一套固定的流程 | 结论经得起追问 |
| 2 | 抽题：留出集每篇论文一道 | 固定种子的考题清单 | 题目不重复数同一家人 |
| 3 | 先写方案 | 预注册：`docs/heldout_plan.md` | 不看了结果再挑指标 |
| 4 | 冻结版本 | git 标签 `v1.0` + 每次考试记录代码版本 | 说得清考的是哪个系统 |
| 5 | 实验怎么记录 | MLflow：参数、指标、标签、追踪、数据集 | 半年后还能说清每个数字怎么来的 |
| 6 | 评测系统的架构 | 数据清单 → 考题清单 → 考试程序 → MLflow | 每一层各管一件事 |
| 7 | 开考，自动检查有效性 | 记录完整、接口故障、泄露审计 | 分数是系统自己的水平 |
| 8 | 结果 | 置信区间、逐题配对检验 | 回答方案里的问题 |
| 9 | 和考前的预期对不对得上 | 对照第 3 步写下的预期 | 知道自己哪里想错了 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 公司里一次正式实验怎么做

### 在干什么
先把流程讲清楚。后面每一步，都是这个流程里的一环。

### 背景
平时改一处、小考一次，随意一点没关系，那是"探索"。但要对外说一个数字（写进论文、报给老板、决定上不上线），就得换一种做法，不然别人一追问就站不住：

- "这个指标是你看了结果以后挑的吧？"
- "你考了几次，是不是挑了最好的一次？"
- "考的到底是哪一版？现在的代码还能跑出这个数吗？"

### 用了什么技术，为什么
一次正式实验的固定流程，以及我们每一环对应的东西：

| 环节 | 做什么 | 我们的做法 |
|---|---|---|
| ① 提出问题 | 这次实验要回答什么 | "最终 agent 是不是比固定流程强" |
| ② 写方案 | 看什么指标、和谁比、用哪些题、怎么算赢 | `docs/heldout_plan.md`，提交进 git（第 3 步） |
| ③ 冻结 | 代码、数据、配置都固定下来 | git 标签 `v1.0`；考题清单带校验码（第 2、4 步） |
| ④ 运行 | 全程自动记录 | `raredx eval run`，记进 MLflow（第 5 步） |
| ⑤ 检查有效性 | 这次实验本身有没有毛病 | 记录完整、接口故障、泄露审计（第 7 步） |
| ⑥ 分析 | 只做方案里写好的比较 | `raredx eval compare`（第 8 步） |
| ⑦ 报告 | 好的坏的都报 | 这本笔记和 README |
| ⑧ 归档 | 别人能照着重跑 | 代码、清单、方案都在 git；记录在 MLflow |

探索和正式实验用的题也不一样：

| 题目 | 多少 | 用来干什么 | 可以看几次 |
|---|---|---|---|
| ci-20 | 20 道 | 小考：改完能不能跑通，花多少钱 | 随便 |
| dev-200 | 200 道 | 开发：比较方案、做决定 | 随便，但分数会偏乐观 |
| **留出集** | 840 道 | 正式评测：对外说的分数 | **一次** |

### 公司里什么情况怎么做
- **模型上线前有一道"离线评测"的关**：新版本必须在一份固定的、开发时碰不到的测试集上达标，才能进入下一步（灰度、A/B 测试）
- **测试集有专人看管**：很多公司不让开发模型的人直接看测试集的题，只能提交系统、拿回分数，防的就是"看多了就贴合了"
- **A/B 测试也是同样的规矩**：实验开始前写好看哪个指标、跑多久、多大差距算赢，不能边跑边看、哪天赢了哪天停

### 面试怎么说
> "Day-to-day iteration ran on a 200-case dev set; the number I report comes from a held-out set no system had touched, run once on a tagged version, under a plan committed before the run: one primary question, fixed comparisons, and only three machine-checked reasons to void a run."

---

## 第 2 步 · 抽题：留出集每篇论文一道

### 在干什么
从留出集里，每篇论文抽一道题，一共 840 道，作为正式考题。

### 背景
留出集有 4,360 个病人，但分布很不均匀：有的论文只报了 1 个病人，有的一篇报了 80 个，常常是同一个病、甚至同一家人。全部都考的话，那一家人的同一种表现会被重复数 80 次，分数被少数几篇论文左右。

dev-200 当初就是"每篇论文一道"抽的（第 06 章），留出集用同样的抽法，两边的分数才好比。

### 用了什么技术，为什么
- **考题清单**（manifest）：一个文本文件，每行一道题的编号和出处论文，提交进 git。病例本身不进 git，清单进
- **固定的随机种子**：谁来抽、什么时候抽，都是同样的 840 道
- **自动检查规则**：清单里的题都在留出集里；每篇论文恰好一道；840 篇论文一篇不漏；和开发用的题没有任何一篇论文重合

### 公司里什么情况怎么做
- **考题清单要冻结并带版本**：换了题，分数就不能和以前比
- **按"来源"抽样而不是按"条数"**：同一个用户、同一篇文档、同一家医院的样本高度相似，要按来源分组，不然会高估自己

### 执行

In [ ]:
from raredx.evals import splits

run("uv run raredx eval check")
rows = splits.read(ROOT / "evals/splits/heldout-papers.tsv")
everything = splits.read(ROOT / "evals/splits/heldout.tsv")
dev = splits.read(ROOT / "evals/splits/design.tsv")
print(f"留出集：{len(everything)} 个病人，{len({p for _, p in everything})} 篇论文")
print(f"正式考题：{len(rows)} 道，来自 {len({p for _, p in rows})} 篇论文")
print("和开发用的病人有没有同一篇论文：", bool({p for _, p in rows} & {p for _, p in dev}))

### 结果怎么看
- 五份清单全部通过规则检查
- 840 道题，840 篇论文，一篇一道
- 和开发用的 6,014 个病人没有任何一篇论文重合：相似病人的病例库只装开发用的病人，所以留出集的题不会在病例库里"找到自己的家人"

### 面试怎么说
> "The held-out test takes one case from each of 840 papers that share no paper with anything used in development; sampling by paper rather than by case keeps one 80-patient family report from being counted 80 times."

---

## 第 3 步 · 先写方案

### 在干什么
开考之前，把"看什么、和谁比、怎么算赢"写成文档，提交进 git。这叫**预注册**。

### 背景
一次考试会产生很多数字：8 个系统、4 种名次、2 种考法。如果考完再挑，总能挑出一组好看的。预注册就是把挑的权利提前用掉：考之前选好，考完照着报。

### 用了什么技术，为什么
方案在 [`docs/heldout_plan.md`](../heldout_plan.md)，要点：

**主要结论只有一个**：只看症状时，最终 agent 的第 1 名命中率是否显著高于固定流程（同一批题逐题比，双侧符号检验，p < 0.05）。

**次要结论**：前 10 名；agent 对传统方法、对大模型直接猜；基因考法里 agent 对基因排名。不管好坏都报。

**只有三种情况重考**，都由程序自动判断，和分数无关：记录不完整；有题目因为接口故障没答；泄露审计没通过。

**不许做的事**：考完改系统再考；挑着报；因为分数不理想换指标、换题。

**考前的预期也写了进去**（第 9 步对答案）。

### 公司里什么情况怎么做
- **临床试验必须预注册**：药企做试验，方案要先在公开网站登记，这个做法就是从那里来的
- **互联网公司的 A/B 实验平台**：开实验时就要填"主要指标"和"预计时长"，平台不让中途改
- **方案里写"预期"**：考完对一下，能看出自己对系统的理解准不准

### 执行
方案是在留出集上跑任何东西之前提交的。从 git 记录里能看出先后：

In [ ]:
run("git log --format='%h %ad %s' --date=format:'%m-%d %H:%M' -- docs/heldout_plan.md evals/splits/heldout-papers.tsv")
run("git tag -n1 v1.0")

### 结果怎么看
- 方案和考题清单在同一次提交里进的 git，时间在所有留出集考试之前
- 之后才打的 `v1.0` 标签（下一步）

### 面试怎么说
> "The plan was committed before the first held-out run: one primary endpoint, the secondary ones, the void-and-rerun rules, and my own predictions — so nothing about how success is judged could be chosen after seeing the scores."

---

## 第 4 步 · 冻结版本

### 在干什么
给代码打一个标签 `v1.0`，所有正式考试都在这一版上跑；每次考试都记下自己用的是哪一版代码。

### 背景
"考的是哪个系统"必须说得清。系统不只是模型，还包括提示词、工具清单、每个工具的写法、防泄露的规则、知识库的版本。半年后有人问"这个 40% 是怎么来的"，要能把那一版原样找回来。

### 用了什么技术，为什么
- **git 标签**：给某一次提交起个名字，以后永远能回到那一版
- **每次考试自动记录代码版本**：记的是"最后一次改动代码的那次提交"。只改文档（比如写这本笔记）不算换版本，所以考试前后写笔记，记录的版本不变
- **有没提交的改动，记录会带 `-dirty`**：笔记默认不用这种记录
- **数据也有版本**：每份外部数据在清单里登记了版本和校验码（第 05 章）；考题清单的校验码也记在每次考试上

### 公司里什么情况怎么做
- **模型注册表**（model registry）：上线的每个版本都登记：代码、数据、配置、评测分数，谁批准的
- **"能复现"是硬要求**：出了事故要回查"当时线上是哪一版、为什么这么回答"
- **版本号一旦对外，就不能再动**：以后改了就是 v1.1，分数另外报告

### 执行

In [ ]:
from raredx.config import settings
from raredx.evals.harness import Harness

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
run("git rev-list -n 1 --abbrev-commit v1.0")
versions = {r["code"] for split in ("heldout-papers", "heldout-papers+genes") for r in harness.runs(split)}
print("留出集上所有考试记录的代码版本：", versions)

### 结果怎么看
- 留出集上的 8 次考试，记录的都是同一个版本，就是打了 `v1.0` 标签的那次提交
- 考试期间我在写这本笔记、提交文档，版本号没有因此改变

### 面试怎么说
> "The evaluated system is a git tag; every run records the last commit that changed code, manifests or dependencies, so documentation commits between runs don't make one system look like two, and an uncommitted change marks the run dirty."

---

## 第 5 步 · 实验怎么记录

### 在干什么
看一次考试到底记下了哪些东西、存在哪。

### 背景
实验记录不是"把分数抄进表格"。一个数字要有用，至少要能回答：这是哪个系统、在哪些题上、哪版代码、花了多少、这次实验本身有没有毛病、每道题具体怎么答的。

### 用了什么技术，为什么
用的是 **MLflow**（第 08 章），行业里最常用的实验记录工具之一。每次考试是一条记录（run），里面分五层：

| 层 | 记了什么 | 例子 |
|---|---|---|
| **参数**（params）：这次考的是什么 | 系统、模型、提示词版本、工具清单、代码版本；考题清单和校验码、数据版本、知识库版本、遮没遮 | `prompt_version = seeded-v1` |
| **指标**（metrics）：结果怎么样 | 第 1/3/5/10 名的命中率和置信区间的上下界；输入输出 token、估算花费、每道题用时 | `disease_top1/mean = 0.405` |
| **有效性**：这次实验有没有毛病 | 接口故障几道；泄露审计读了多少次工具回答、漏出几次 | 标签 `leak_audit = passed` |
| **追踪**（traces）：每道题的全过程 | 题目、每次工具调用的输入输出、每次模型调用、最终答案、每道题判对判错 | 一道题一条，可以在网页上逐步看 |
| **数据集**（dataset） | 这份考题在 MLflow 里的登记，带校验码 | `heldout-papers` |

这些都存在本机（`mlflow.db` 和 `mlruns/`），不上传到任何地方。

### 公司里什么情况怎么做
- **实验记录工具是标配**：MLflow、Weights & Biases，做大模型应用的还有 LangSmith、Langfuse。记的东西大同小异：参数、指标、产物、追踪
- **"谁都能查"比"我记得"重要**：同事、半年后的自己，都要能不问人就查到一个数字的来龙去脉
- **参数和指标分开**：参数是你设的，指标是量出来的。混在一起，以后没法按条件筛选、比较

### 执行
调出 dev-200 上一次 agent 考试的记录（第 20 章带全部工具的那次），看每一层长什么样。

In [ ]:
def latest(name, split):
    # 带 -dirty 的记录，只有复核过、打了标签的才用
    for r in harness.runs(split):
        if r["name"] == name and (not r["code"].endswith("-dirty")
                                  or "code_reviewed" in harness.client.get_run(r["run_id"]).data.tags):
            return r

record = latest("seeded-agent", "dev-200")
data = harness.client.get_run(record["run_id"]).data
print("【参数】")
for key in ("system", "model", "prompt_version", "tools", "code", "manifest", "manifest_sha256", "hpo_release", "patient_bank", "knowledge_mask"):
    print(f"  {key:<16} {str(data.params.get(key))[:70]}")
print("【指标】")
for key in ("disease_top1/mean", "disease_top1/low", "disease_top1/high", "disease_top10/mean", "usage/input_tokens", "usage/cost_usd_estimate", "usage/seconds_per_case"):
    print(f"  {key:<24} {data.metrics.get(key)}")
print("【有效性】")
for key in ("errors/model_api", "leak/tool_outputs_read", "leak/own_paper_shown", "leak/hidden_disease_shown"):
    print(f"  {key:<26} {data.metrics.get(key)}")
print(f"  {'标签 leak_audit':<26} {data.tags.get('leak_audit')}")
traces = harness.traces(record["run_id"])
print(f"【追踪】 {len(traces)} 条，每道题一条；第一条里有 {len(traces[0].data.spans)} 步")

### 结果怎么看
- **参数**说清了考的是什么：哪个系统、哪版提示词、哪些工具、哪版代码、哪份题（带校验码）、哪版知识库、遮了什么
- **指标**里，命中率带着置信区间的上下界；花费和用时也在
- **有效性**：接口故障 0 道；泄露审计读了一万多次工具回答，漏出 0 次
- **追踪**：200 道题每道一条，一条里有一百多步，可以逐步回看

想看网页版：在项目目录运行 `uv run mlflow ui --port 5050`，浏览器打开 http://127.0.0.1:5050 。

### 面试怎么说
> "Every run is one MLflow record: parameters say what was tested — system, prompt version, tools, code commit, manifest and its checksum; metrics hold recall with Wilson bounds, tokens and cost; validity counters hold model-API failures and the leak audit; and a full trace per case lets anyone replay how an answer was reached."

---

## 第 6 步 · 评测系统的架构

### 在干什么
把前面二十章搭起来的评测系统画成一张图，看每一层各管什么。

### 背景
评测系统本身也是一个软件，而且是这个项目里最重要的一个：agent 换了多少版，尺子一直是这一把。它的每一层只管一件事，换掉其中一层不影响别的。

### 用了什么技术，为什么

```
① 数据清单   evals/datasets.toml           每份外部数据的版本、地址、许可证、校验码
                    │  raredx data fetch / build
                    ▼
② 数据       data/raw  →  data/processed    原始下载（核对校验码）→ 统一格式的病例
                    │
③ 考题清单   evals/splits/*.tsv            哪次考试考哪些题；冻结在 git 里
                    │
                    ▼
④ 考试程序   raredx eval run  →  Harness
                    │   对每道题：
                    │     登记"遮住这道题的论文"
                    │     → 被考的系统作答（工具调用、模型调用，每一步都记成追踪）
                    │     → 判分（第 1/3/5/10 名对不对）
                    │   考完自动检查：记录完整吗？有接口故障吗？泄露审计过了吗？
                    ▼
⑤ 实验记录   MLflow（本机）                 参数、指标、有效性、每道题的追踪、考题登记
                    │
                    ▼
⑥ 分析和报告  raredx eval compare / runs   逐题配对比较、列出所有考试
             这些笔记                        只读记录，不花钱
```

被考的系统（传统方法、固定流程、agent……）都是"给一道题，返回一串候选诊断"这同一个接口。考试程序不关心里面是什么，所以所有系统用的是**同一把尺子、同一套防泄露规则、同一套检查**。

| 层 | 公司里通常叫什么 |
|---|---|
| ① 数据清单 | 数据目录、数据集注册表 |
| ③ 考题清单 | 评测集、"黄金集"（golden set） |
| ④ 考试程序 | 评测流水线（eval harness / eval pipeline） |
| ④ 里的自动检查 | 评测的质量门槛（validity checks） |
| ⑤ 实验记录 | 实验追踪平台（experiment tracking） |
| ⑥ 分析 | 评测报告、看板 |

### 公司里什么情况怎么做
- **评测系统和被评测的系统分开**：评测代码不依赖某一个模型或 agent，换系统不用改尺子
- **评测进 CI**：每次改代码自动跑一个小评测集（我们的 ci-20），防止改坏
- **尺子本身也要测**：第 07 章用"标准答案系统必须 100 分"来验证判分程序；泄露审计、接口故障检查都有自己的测试

### 面试怎么说
> "The evaluation stack has six layers, each with one job: a dataset registry with checksums, frozen manifests in git, a harness that masks, runs, scores and then checks its own validity, MLflow for the record, and paired-comparison tooling on top. Every system, from a classical ranker to the agent, sits behind the same predict interface, so they all meet the same ruler."

---

## 第 7 步 · 开考，自动检查有效性

### 在干什么
按方案考 8 次。每次考完，程序自动做三项检查，任何一项不过，这次考试作废。

### 背景
分数只有在考试本身没毛病的时候才有意义。第 18、20 章踩过的三个坑，现在都是自动检查：

| 检查 | 防的是什么 | 哪一章加的 |
|---|---|---|
| 记录完整 | 机器太忙时有的题没记下来，分数就不是全部题的分数 | 第 08 章 |
| 接口故障 | 欠费、限流、超时被当成"系统答错" | 第 20 章 |
| 泄露审计 | 工具的回答里漏出题目自己的论文，或只从这篇论文才知道的病 | 第 18 章 |

### 执行

In [ ]:
RUNS = [("大模型直接猜", "llm-direct", "heldout-papers"),
        ("传统方法：按症状排名", "phenotype-rank", "heldout-papers"),
        ("找相似病人", "similar-patients", "heldout-papers"),
        ("先排名再逐条核对", "rank-then-check", "heldout-papers"),
        ("固定流程", "fixed-pipeline", "heldout-papers"),
        ("最终 agent", "seeded-agent", "heldout-papers"),
        ("基因考法：按基因排名", "gene-rank", "heldout-papers+genes"),
        ("基因考法：最终 agent", "seeded-agent", "heldout-papers+genes")]
records, total = {}, 0.0
for label, name, split in RUNS:
    records[label] = record = latest(name, split)
    d = harness.client.get_run(record["run_id"]).data
    m = d.metrics
    usd = ((m.get("usage/input_tokens", 0) - m.get("usage/cached_input_tokens", 0)) * 0.15
           + m.get("usage/cached_input_tokens", 0) * 0.003 + m.get("usage/output_tokens", 0) * 0.60) / 1e6
    total += usd
    print(f"{label:<12} 应考 {m['n_cases']:.0f} 作答 {m['n_answered']:.0f}  接口故障 {m.get('errors/model_api', 0):.0f}  "
          f"泄露审计 {d.tags.get('leak_audit'):<15} 读了 {m.get('leak/tool_outputs_read', 0):>6,.0f} 次工具回答  约 ${usd:.2f}")
print(f"\n合计约 ${total:.2f}（非高峰价），约 ¥{total * 7.2:.0f}")

### 结果怎么看
- **8 次考试全部有效**，没有一次需要重考：题目全部作答，接口故障 0 道，泄露审计全部通过
- 两次 agent 考试各读了五万多次工具回答，漏出 0 次
- "大模型直接猜"不调用工具，审计如实显示"没有工具回答可查"，而不是"通过"
- 一共花了约 ¥39，和方案里估的一样，在 ¥50 的上限以内

### 面试怎么说
> "All eight held-out runs were valid on the first attempt: every case answered, no model-API failures, and the leak audit clean over more than fifty thousand tool outputs per agent run."

---

## 第 8 步 · 结果

### 在干什么
按方案报告：每个系统的命中率和置信区间，事先定好的逐题比较，以及两项分开看的分析。

### 执行
先读出 8 次考试里每道题的对错（要读全部记录，大约几分钟），后面几格都用它。

In [ ]:
from raredx.evals import scoring

hits = {label: {k: harness.hits(record["run_id"], k) for k in (1, 10)} for label, record in records.items()}
print(f"{'':<14}{'第 1 名':>22}{'前 10 名':>24}")
for label, record in records.items():
    m = harness.client.get_run(record["run_id"]).data.metrics
    cells = [f"{m[f'disease_top{k}/mean']:.1%} [{m[f'disease_top{k}/low']:.1%}–{m[f'disease_top{k}/high']:.1%}]" for k in (1, 10)]
    print(f"{label:<12}{cells[0]:>26}{cells[1]:>26}")

### 结果怎么看

| 留出集，840 道题 | 第 1 名 | 前 10 名 |
|---|---|---|
| 大模型直接猜 | 14.3% [12.1–16.8] | 29.0% [26.1–32.2] |
| 传统方法：按症状排名 | 17.5% [15.1–20.2] | 44.4% [41.1–47.8] |
| 找相似病人 | 27.9% [24.9–31.0] | 50.2% [46.9–53.6] |
| 先排名再逐条核对 | 32.1% [29.1–35.4] | 51.5% [48.2–54.9] |
| 固定流程 | 36.8% [33.6–40.1] | 51.5% [48.2–54.9] |
| **最终 agent** | **42.4% [39.1–45.8]** | **62.0% [58.7–65.2]** |
| *基因考法（839 道）*：按基因排名 | 59.5% [56.1–62.7] | 80.0% [77.1–82.5] |
| *基因考法*：**最终 agent** | **65.9% [62.6–69.0]** | **81.4% [78.6–83.9]** |

方括号里是 95% 置信区间。840 道题让区间收窄到正负 3 个百分点左右，dev-200 上是正负 7。

下面是方案里事先定好的逐题比较。

In [ ]:
def compare(a, b, k):
    wins = sum(hits[a][k][c] and not hits[b][k][c] for c in hits[a][k])
    losses = sum(hits[b][k][c] and not hits[a][k][c] for c in hits[a][k])
    return f"{wins:>4} 比 {losses:<3} {scoring.p_value(scoring.sign_test(wins, losses))}"

print("【主要结论】只看症状，第 1 名：最终 agent 对固定流程   ", compare("最终 agent", "固定流程", 1))
print("\n【次要结论】")
print("只看症状，前 10 名：最终 agent 对固定流程             ", compare("最终 agent", "固定流程", 10))
print("只看症状，第 1 名：最终 agent 对传统方法               ", compare("最终 agent", "传统方法：按症状排名", 1))
print("只看症状，第 1 名：最终 agent 对大模型直接猜           ", compare("最终 agent", "大模型直接猜", 1))
print("基因考法，第 1 名：最终 agent 对按基因排名             ", compare("基因考法：最终 agent", "基因考法：按基因排名", 1))
print("基因考法，前 10 名：最终 agent 对按基因排名            ", compare("基因考法：最终 agent", "基因考法：按基因排名", 10))

### 结果怎么看

**主要结论成立**：只看症状时，最终 agent 的第 1 名命中率显著高于固定流程。840 道题里，只有 agent 对的 57 道，只有固定流程对的 10 道，p < 0.001。

次要结论也都成立：

| 比较 | 只有 agent 对 | 只有对方对 | |
|---|---|---|---|
| 对固定流程，前 10 名 | 94 | 6 | p < 0.001 |
| 对传统方法，第 1 名 | 217 | 8 | p < 0.001 |
| 对大模型直接猜，第 1 名 | 257 | 21 | p < 0.001 |
| 基因考法：对基因排名，第 1 名 | 59 | 5 | p < 0.001 |
| 基因考法：对基因排名，前 10 名 | 15 | 3 | p = 0.008 |

### 分开看 ①：能答的题和注定答不对的题

方案里说过，有 122 道题的病只在题目自己的论文里描述过（或者 HPO 里根本没有），按防泄露的规则谁都答不对。把它们分开算。

In [ ]:
from raredx.data import cases as case_store, sources
from raredx.evals import splits
from raredx.knowledge import hpo as hpo_knowledge

registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
cases = {c.id: c for c in case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))}
exam = splits.read(ROOT / "evals/splits/heldout-papers.tsv")
unanswerable = {cid for cid, pmid in exam if cases[cid].disease_id not in hpo.names
                or cases[cid].disease_id in hpo.known_only_from(frozenset({pmid}))}
print(f"注定答不对的题：{len(unanswerable)} 道\n")
print(f"{'':<14}{'能答的题：第 1 名':>14}{'前 10 名':>10}{'注定答不对的题里答对了':>16}")
for label in records:
    ok = [c for c in hits[label][1] if c not in unanswerable]
    rates = [sum(hits[label][k][c] for c in ok) / len(ok) for k in (1, 10)]
    lucky = sum(hits[label][10][c] for c in hits[label][10] if c in unanswerable)
    print(f"{label:<12}{rates[0]:>16.1%}{rates[1]:>14.1%}{lucky:>16} 道")

### 结果怎么看
- **在能答的 718 道题上，最终 agent 第 1 名 49.6%、前 10 名 72.6%**；有基因时 77.1% 和 95.3%
- **注定答不对的 122 道，所有用工具的系统一道都没答对**，agent 也是 0 道。这是对防泄露的又一次验证：按规则这些题不该答得出来，实际也确实答不出来。如果某个系统在这些题上答对了不少，那才是出了问题
- 只有"大模型直接猜"在这些题上有 5 道把正确答案放进了前 10 名（其中 2 道排第 1）：那是模型自己的记忆或者运气，工具管不着

### 分开看 ②：按论文发表年份

越新的论文，大模型越不可能在训练时读到。如果分数靠的是"模型背过论文"，新论文上应该掉得特别多。

In [ ]:
import csv

years = {row["pmid"]: int(row["year"]) for row in csv.DictReader(open(ROOT / "docs/heldout_pubmed_years.tsv"), delimiter="\t")}
periods = [("2010 年及以前", 0, 2010), ("2011–2017", 2011, 2017), ("2018–2021", 2018, 2021), ("2022–2023", 2022, 2023), ("2024 年以后", 2024, 2100)]
shown = ("大模型直接猜", "传统方法：按症状排名", "固定流程", "最终 agent")
print(f"{'发表年份':<10}{'题数':>4}{'注定答不对':>8}    能答的题上的第 1 名：" + "".join(f"{s[:6]:>10}" for s in shown))
for label, lo, hi in periods:
    ids = [cid for cid, pmid in exam if lo <= years[pmid] <= hi]
    ok = [c for c in ids if c not in unanswerable]
    rates = "".join(f"{sum(hits[s][1][c] for c in ok) / len(ok):>12.1%}" for s in shown)
    print(f"{label:<10}{len(ids):>6}{(len(ids) - len(ok)) / len(ids):>10.0%}                    {rates}")

### 结果怎么看

| 发表年份 | 题数 | 注定答不对 | 大模型直接猜 | 传统方法 | 固定流程 | 最终 agent |
|---|---|---|---|---|---|---|
| 2010 年及以前 | 158 | 7% | 27.2% | 26.5% | 55.1% | 62.6% |
| 2011–2017 | 245 | 4% | 18.4% | 23.5% | 46.6% | 53.4% |
| 2018–2021 | 205 | 8% | 12.2% | 18.1% | 39.4% | 45.2% |
| 2022–2023 | 125 | 25% | 8.5% | 16.0% | 30.9% | 38.3% |
| 2024 年以后 | 107 | 49% | 7.3% | 7.3% | 29.1% | 32.7% |

- **越新的论文越难**：2024 年以后的论文，将近一半报的是首次发现的病，按规则答不出来。就算能答的，分数也低得多
- **"大模型直接猜"掉得很厉害**（27% → 7%），但**不用模型的传统方法掉得一样多**（26.5% → 7.3%）。所以主要原因是"老病研究得多、资料全"，不能单独算在"模型背过论文"头上
- **agent 对固定流程的领先，每个年份段都在**（多 4–8 个百分点）。它的优势不是靠记住了某些老病例

这一项是探索性的，不用来下主要结论。要干净地量出"模型记忆"占多少分，得找同一种病在模型训练前后各发表的病例来比，这里的数据做不到。

### 公司里什么情况怎么做
- **主要结论、次要结论、探索性分析要分开标明**：探索性分析是"看到了一个现象"，不是"证明了一件事"
- **报告置信区间和逐题比较，不只报一个百分比**：42.4% 对 36.8% 看着差不多，逐题比是 57 比 10
- **把"按规则不可能答对的题"单独拿出来看**：它们该是 0 分。不是 0，说明评测漏了

### 面试怎么说
> "On 840 held-out cases, one per paper, the final agent gets the confirmed disease first in 42.4% [39.1–45.8] against 36.8% for the fixed pipeline — paired, 57 cases to 10, p < 0.001 — and in the top ten in 62.0% against 51.5%. With simulated sequencing it reaches 65.9% against 59.5% for a no-model gene ranking. On the 122 cases that strict masking makes unanswerable, every tool-using system scores zero, as it should."

---

## 第 9 步 · 和考前的预期对不对得上

### 在干什么
把方案里考前写下的预期，和实际结果放在一起看。

### 背景
考前我写了三条预期（[方案](../heldout_plan.md)第 7 节）。写预期不是为了猜对，是为了考完能看出：我对这个系统的理解，哪里是对的，哪里错了。

### 结果怎么看

| 考前的预期 | 实际 | 对吗 |
|---|---|---|
| 各系统的绝对分数和 dev-200 差不多，可能略高 | 对照组都在误差范围内；agent 42.4% 对 dev-200 的 40.5% | ✅ |
| agent 对固定流程的领先会缩小（dev-200 上 5 个百分点，可能缩到 2–3 个） | **没有缩小**：第 1 名多 5.6 个百分点，前 10 名多 10.5 个（dev-200 上是 5 和 4） | ❌ 我想错了 |
| 基因考法的合并最可能缩水（dev-200 上比基因排名多 5 个百分点） | **没有缩水**：多 6.4 个百分点 | ❌ 我想错了 |

**为什么我想错了**

我担心的是对开发集过拟合：看着 200 道题做了很多决定，分数会虚高。实际没有发生，回头看有两个原因：

1. **我们做的决定是"结构性"的，不是"调参数"**。留哪个工具、删哪个、开局带不带证据，每个决定只有"是"或"否"，加起来也就十来个。过拟合严重的情况，是在几十上百个数字参数里挑最好的一组
2. **每个决定都有逐题比较的证据**，不显著的不采用（比如"查疾病介绍"、三种判断方法）。这本身就是在防过拟合

但这不等于"以后可以不留留出集"。考之前我并不知道会不会缩水，只有留出集能回答这个问题。这次是没缩水，下次未必。

**前 10 名的领先反而大了**（dev-200 上多 4 个百分点，留出集上多 10.5 个）。200 道题的误差本来就大，dev-200 上那个数很可能偏低了。这也说明小样本上的差距，偏高偏低都有可能。

### 公司里什么情况怎么做
- **留出集的结果和开发集差很多，是最需要警惕的信号**：通常意味着过拟合，或者两份数据分布不一样
- **预测错了也要写下来**：只记预测对的，下次还会犯同样的判断错误

### 面试怎么说
> "I had predicted the agent's lead would shrink on held-out data, since every design decision had been made looking at the dev set. It did not — 5.6 points at first place against 5 on dev — which I put down to the decisions being few, structural, and each backed by a paired test. I recorded the wrong prediction next to the result."

---

## 本章小结

| 做了什么 | 结果 |
|---|---|
| 抽题 | 留出集 840 篇论文各一道；和开发用的病人没有一篇论文重合 |
| 先写方案 | 一个主要结论、五个次要结论、三条重考规则、考前的预期，都在开考前提交 |
| 冻结版本 | `v1.0`；8 次考试记录的都是这一版 |
| 考试 | 8 次全部有效，没有重考；约 ¥39 |
| **主要结论** | **最终 agent 第 1 名 42.4%，固定流程 36.8%；逐题比 57 比 10，p < 0.001，成立** |
| 前 10 名 | 62.0% 对 51.5%；94 比 6 |
| 基因考法 | 65.9% 对基因排名的 59.5%；59 比 5 |
| 能答的题上 | 第 1 名 49.6%，前 10 名 72.6%；有基因时 77.1% 和 95.3% |
| 防泄露的验证 | 122 道注定答不对的题，所有用工具的系统都是 0 道 |
| 考前预期 | 三条里错了两条：领先没有缩小 |

这一章**没有做**的（留到以后，用同一份 840 道题）：和 Exomiser、LIRICAL、PubCaseFinder 这些通用诊断工具比；和通用大模型（GPT、Claude、Gemini）比；和已发表的 DeepRare 比；在 RareBench 这些别人论文用的数据集上考。

## 面试题

1. **How do you know your agent's numbers are real?**
   > "They come from a held-out set no system had touched, run once on a tagged version under a pre-committed plan. Every run passed three machine checks — complete record, no model-API failures, a leak audit over every tool output — and the 122 cases that masking makes unanswerable score zero for every tool-using system."

2. **Walk me through how you record an experiment.**
   > "One MLflow run per evaluation: parameters for what was tested, including code commit and the manifest's checksum; metrics for recall with Wilson bounds, tokens and cost; validity counters; and a full trace per case. A number can be traced back to its code, data and every tool call."

3. **What is pre-registration and why bother outside a clinical trial?**
   > "Deciding the primary metric, the comparisons and the rerun rules before seeing results. An eight-system, two-track evaluation yields dozens of numbers; without a plan you can always find a flattering subset."

4. **Did the held-out result surprise you?**
   > "Yes — I expected the lead over the fixed pipeline to shrink and wrote that down beforehand. It held. I'd still never skip the held-out run: before it, I couldn't have known."

**下一章 22 · 做成服务**：分数有了，接下来把 agent 做成一个能演示的服务：一个网页，输入症状，实时看 agent 每一步查了什么、为什么这么判断，最后给出带出处的诊断。